# 09 — Siamese U-Net Fine-Tuning

**Question:** How much does fine-tuning improve the pretrained model?

**Pipeline connection:** Best weights saved to `ml/models/siamese_unet_best.pt` for use by `ml/inference/damage.py`.

Two-stage training: freeze backbone and train the head first (Stage 1), then unfreeze
the top encoder blocks and fine-tune end-to-end with a lower learning rate (Stage 2).
Everything is logged to MLflow for comparison.

In [ ]:
import random
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import mlflow
import numpy as np
import rasterio
import torch
import torch.nn as nn
import torch.optim as optim
from rasterio.windows import from_bounds
from sklearn.metrics import f1_score, precision_score, recall_score
from torch.utils.data import DataLoader, Dataset

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"
MODEL_DIR = PROJECT_ROOT / "ml" / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# AOI and observation dates
AOI = [-105.23, 39.915, -105.12, 39.98]
DATES = ["2021-11", "2022-01", "2022-06", "2023-06", "2024-06"]
PATCH_SIZE = 64
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

## Build the training dataset

Each sample is a `(pre_patch, post_patch, label)` triplet. Pre = Nov 2021 VV,
post = Jan 2022 VV. Labels come from the parcel damage status (1 = destroyed, 0 = survived).
SAR-valid augmentations only: random flips and 90-degree rotations (no color jitter
since SAR intensity has physical meaning).

In [ ]:
class SarAugmentation:
    """SAR-valid augmentations: random flip + 90-degree rotation.
    
    We avoid color jitter / brightness changes because SAR backscatter
    values (dB) have physical meaning -- altering them would change the
    signal the model is supposed to learn.
    """

    def __call__(self, pre: np.ndarray, post: np.ndarray):
        # Random horizontal flip
        if random.random() > 0.5:
            pre = np.flip(pre, axis=-1).copy()
            post = np.flip(post, axis=-1).copy()
        # Random vertical flip
        if random.random() > 0.5:
            pre = np.flip(pre, axis=-2).copy()
            post = np.flip(post, axis=-2).copy()
        # Random 90-degree rotation (0, 1, 2, or 3 times)
        k = random.randint(0, 3)
        if k > 0:
            pre = np.rot90(pre, k, axes=(-2, -1)).copy()
            post = np.rot90(post, k, axes=(-2, -1)).copy()
        return pre, post


class MarshallFireDataset(Dataset):
    """PyTorch dataset returning (pre_tensor, post_tensor, label) for each parcel.
    
    Patches are extracted from COG rasters at parcel centroids.
    """

    def __init__(
        self,
        parcels: gpd.GeoDataFrame,
        pre_raster_path: Path,
        post_raster_path: Path,
        patch_size: int = 64,
        augment: bool = False,
    ):
        self.parcels = parcels.reset_index(drop=True)
        self.pre_raster_path = pre_raster_path
        self.post_raster_path = post_raster_path
        self.patch_size = patch_size
        self.augment = SarAugmentation() if augment else None

        # Pre-compute centroids in raster CRS
        with rasterio.open(self.pre_raster_path) as src:
            self.raster_crs = src.crs
            self.raster_res = src.res[0]  # pixel size in meters

        centroids = self.parcels.to_crs(self.raster_crs).geometry.centroid
        self.cx = centroids.x.values
        self.cy = centroids.y.values
        self.labels = self.parcels["damaged"].values.astype(np.float32)

    def __len__(self):
        return len(self.parcels)

    def _extract_patch(self, raster_path: Path, cx: float, cy: float) -> np.ndarray:
        """Extract a patch_size x patch_size window centered on (cx, cy)."""
        half = (self.patch_size / 2) * self.raster_res
        window_bounds = (cx - half, cy - half, cx + half, cy + half)

        with rasterio.open(raster_path) as src:
            window = from_bounds(*window_bounds, transform=src.transform)
            patch = src.read(1, window=window)

        # Pad if patch is smaller than expected (edge parcels)
        if patch.shape != (self.patch_size, self.patch_size):
            padded = np.zeros((self.patch_size, self.patch_size), dtype=patch.dtype)
            padded[: patch.shape[0], : patch.shape[1]] = patch
            patch = padded

        return patch.astype(np.float32)

    def __getitem__(self, idx):
        cx, cy = self.cx[idx], self.cy[idx]

        pre = self._extract_patch(self.pre_raster_path, cx, cy)
        post = self._extract_patch(self.post_raster_path, cx, cy)

        # Add channel dimension: (1, H, W)
        pre = pre[np.newaxis, :, :]
        post = post[np.newaxis, :, :]

        if self.augment is not None:
            pre, post = self.augment(pre, post)

        label = self.labels[idx]

        return (
            torch.from_numpy(pre),
            torch.from_numpy(post),
            torch.tensor(label, dtype=torch.float32),
        )


# ----- Load parcels with damage labels -----
parcels = gpd.read_parquet(DATA_DIR / "tabular" / "parcel_labels.parquet")
print(f"Total parcels: {len(parcels)}")
print(f"Damaged: {parcels['damaged'].sum()}, Survived: {(~parcels['damaged'].astype(bool)).sum()}")

## Spatial block split -- prevent data leakage

Random splits let the model memorize spatial autocorrelation.
Instead, split by longitude: west half = train, east half = val.
This simulates applying the model to a new neighborhood.

In [ ]:
# Compute centroid longitude for splitting
parcels_4326 = parcels.to_crs(epsg=4326)
parcels["centroid_lon"] = parcels_4326.geometry.centroid.x

lon_median = parcels["centroid_lon"].median()
print(f"Split longitude (median): {lon_median:.4f}")

train_mask = parcels["centroid_lon"] <= lon_median
val_mask = parcels["centroid_lon"] > lon_median

parcels_train = parcels[train_mask].copy()
parcels_val = parcels[val_mask].copy()

print(f"Train: {len(parcels_train)} parcels ({parcels_train['damaged'].sum()} damaged)")
print(f"Val:   {len(parcels_val)} parcels ({parcels_val['damaged'].sum()} damaged)")

# ----- Create datasets and dataloaders -----
pre_raster = DATA_DIR / "processed" / "sar" / "vv_2021-11_db.tif"
post_raster = DATA_DIR / "processed" / "sar" / "vv_2022-01_db.tif"

train_ds = MarshallFireDataset(
    parcels_train, pre_raster, post_raster,
    patch_size=PATCH_SIZE, augment=True,
)
val_ds = MarshallFireDataset(
    parcels_val, pre_raster, post_raster,
    patch_size=PATCH_SIZE, augment=False,
)

BATCH_SIZE = 16
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# Sanity check: one batch
pre_batch, post_batch, label_batch = next(iter(train_loader))
print(f"Batch shapes: pre={pre_batch.shape}, post={post_batch.shape}, labels={label_batch.shape}")

## Define loss function and training helpers

Combined BCE + Dice loss with `pos_weight=3.4` to handle class imbalance
(destroyed parcels are roughly 23% of the dataset, so 1/0.23 is approximately 3.4,
upweighting the minority class). Dice loss adds a region-level overlap signal that
BCE alone misses.

In [ ]:
class CombinedLoss(nn.Module):
    """BCE + Dice loss with positive class weighting."""

    def __init__(self, pos_weight: float = 3.4, bce_weight: float = 0.5):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss(
            pos_weight=torch.tensor([pos_weight])
        )
        self.bce_weight = bce_weight

    @staticmethod
    def dice_loss(logits: torch.Tensor, targets: torch.Tensor, smooth: float = 1.0):
        probs = torch.sigmoid(logits)
        intersection = (probs * targets).sum()
        return 1.0 - (2.0 * intersection + smooth) / (
            probs.sum() + targets.sum() + smooth
        )

    def forward(self, logits: torch.Tensor, targets: torch.Tensor):
        bce_loss = self.bce(logits, targets)
        d_loss = self.dice_loss(logits, targets)
        return self.bce_weight * bce_loss + (1 - self.bce_weight) * d_loss


def run_epoch(model, loader, criterion, optimizer, device, train=True):
    """Run one training or validation epoch. Returns avg loss."""
    model.train() if train else model.eval()
    total_loss = 0.0
    n_batches = 0

    context = torch.no_grad() if not train else torch.enable_grad()
    with context:
        for pre, post, labels in loader:
            pre = pre.to(device)
            post = post.to(device)
            labels = labels.to(device)

            # Forward: Siamese network takes stacked pre/post
            x = torch.cat([pre, post], dim=1)  # (B, 2, H, W)
            logits = model(x).squeeze(1)  # (B,)

            loss = criterion(logits, labels)

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item()
            n_batches += 1

    return total_loss / max(n_batches, 1)


def run_evaluation(model, loader, device, threshold=0.5):
    """Compute F1, precision, recall on a data loader."""
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for pre, post, labels in loader:
            pre = pre.to(device)
            post = post.to(device)
            x = torch.cat([pre, post], dim=1)
            logits = model(x).squeeze(1)
            probs = torch.sigmoid(logits).cpu().numpy()
            all_preds.extend((probs >= threshold).astype(int))
            all_labels.extend(labels.numpy().astype(int))

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)

    return {
        "f1": f1_score(all_labels, all_preds, zero_division=0),
        "precision": precision_score(all_labels, all_preds, zero_division=0),
        "recall": recall_score(all_labels, all_preds, zero_division=0),
    }


print("Loss and helpers defined.")

## Stage 1 -- Freeze backbone, train head only (10 epochs)

With the encoder frozen, only the decoder/classifier head learns.
This prevents catastrophic forgetting of pretrained features while
the randomly-initialized head catches up.

In [ ]:
# ----- Load pretrained Siamese U-Net -----
# Architecture: ResNet50 encoder (2-channel input for VV) + classifier head
from torchvision.models import resnet50


class SiameseUNet(nn.Module):
    """Siamese U-Net for bi-temporal change detection.
    
    Takes 2-channel input (pre VV + post VV stacked), encodes with a
    shared ResNet50 backbone, and outputs a single damage probability.
    """

    def __init__(self):
        super().__init__()
        backbone = resnet50(weights=None)
        # Modify first conv to accept 2 channels (pre VV + post VV)
        backbone.conv1 = nn.Conv2d(2, 64, kernel_size=7, stride=2, padding=3, bias=False)
        
        self.layer0 = nn.Sequential(backbone.conv1, backbone.bn1, backbone.relu, backbone.maxpool)
        self.layer1 = backbone.layer1  # 256 channels
        self.layer2 = backbone.layer2  # 512 channels
        self.layer3 = backbone.layer3  # 1024 channels
        self.layer4 = backbone.layer4  # 2048 channels

        # Global average pool + binary classifier
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(2048, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 1),
        )

    def forward(self, x):
        x = self.layer0(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.gap(x)
        return self.classifier(x)


model = SiameseUNet().to(DEVICE)

# Load pretrained weights if available
pretrained_path = MODEL_DIR / "siamese_unet_pretrained.pt"
if pretrained_path.exists():
    model.load_state_dict(torch.load(pretrained_path, map_location=DEVICE, weights_only=True))
    print(f"Loaded pretrained weights from {pretrained_path}")
else:
    print("No pretrained weights found -- training from scratch.")

# ----- Freeze encoder -----
for name, param in model.named_parameters():
    if not name.startswith("classifier"):
        param.requires_grad = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")

# ----- Stage 1 training -----
criterion = CombinedLoss(pos_weight=3.4).to(DEVICE)
optimizer_s1 = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3
)

STAGE1_EPOCHS = 10
history = {"train_loss": [], "val_loss": [], "val_f1": [], "val_precision": [], "val_recall": []}

print(f"\n{'Epoch':>5} {'Train Loss':>10} {'Val Loss':>10} {'Val F1':>8} {'Val P':>8} {'Val R':>8}")
print("-" * 55)

for epoch in range(STAGE1_EPOCHS):
    train_loss = run_epoch(model, train_loader, criterion, optimizer_s1, DEVICE, train=True)
    val_loss = run_epoch(model, val_loader, criterion, None, DEVICE, train=False)
    metrics = run_evaluation(model, val_loader, DEVICE)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_f1"].append(metrics["f1"])
    history["val_precision"].append(metrics["precision"])
    history["val_recall"].append(metrics["recall"])

    print(
        f"{epoch+1:5d} {train_loss:10.4f} {val_loss:10.4f}"
        f" {metrics['f1']:8.4f} {metrics['precision']:8.4f} {metrics['recall']:8.4f}"
    )

print(f"\nStage 1 best val F1: {max(history['val_f1']):.4f}")

## Stage 2 -- Unfreeze top encoder blocks (20 epochs)

Now that the head has converged, unfreeze `layer3` and `layer4` so the
encoder can adapt its features to SAR data. Lower learning rate + cosine
schedule prevent destructive weight updates. Early stopping (patience=5)
saves the best checkpoint.

In [ ]:
# ----- Unfreeze layer3 and layer4 -----
for name, param in model.named_parameters():
    if name.startswith("layer3") or name.startswith("layer4") or name.startswith("classifier"):
        param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params after unfreeze: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")

# ----- Stage 2 optimizer + scheduler -----
STAGE2_EPOCHS = 20
optimizer_s2 = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4
)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer_s2, T_max=STAGE2_EPOCHS)

# ----- Early stopping state -----
best_val_f1 = max(history["val_f1"]) if history["val_f1"] else 0.0
patience = 5
patience_counter = 0
best_model_path = MODEL_DIR / "siamese_unet_best.pt"

# Mark the boundary between Stage 1 and Stage 2
stage_boundary = len(history["train_loss"])

print(f"\n{'Epoch':>5} {'Train Loss':>10} {'Val Loss':>10} {'Val F1':>8} {'LR':>10} {'ES':>4}")
print("-" * 55)

for epoch in range(STAGE2_EPOCHS):
    train_loss = run_epoch(model, train_loader, criterion, optimizer_s2, DEVICE, train=True)
    val_loss = run_epoch(model, val_loader, criterion, None, DEVICE, train=False)
    metrics = run_evaluation(model, val_loader, DEVICE)
    current_lr = scheduler.get_last_lr()[0]
    scheduler.step()

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_f1"].append(metrics["f1"])
    history["val_precision"].append(metrics["precision"])
    history["val_recall"].append(metrics["recall"])

    # Early stopping check
    if metrics["f1"] > best_val_f1:
        best_val_f1 = metrics["f1"]
        patience_counter = 0
        torch.save(model.state_dict(), best_model_path)
        marker = "*"
    else:
        patience_counter += 1
        marker = ""

    print(
        f"{stage_boundary + epoch + 1:5d} {train_loss:10.4f} {val_loss:10.4f}"
        f" {metrics['f1']:8.4f} {current_lr:10.6f} {patience_counter:3d} {marker}"
    )

    if patience_counter >= patience:
        print(f"\nEarly stopping at epoch {stage_boundary + epoch + 1}.")
        break

print(f"\nBest val F1: {best_val_f1:.4f}")
print(f"Model saved to {best_model_path}")

## Plot training curves

Vertical dashed line marks the freeze/unfreeze boundary between Stage 1 and Stage 2.
Look for:
- Stage 1: loss dropping quickly (head learning fast with frozen features)
- Stage 2: slower but continued improvement as encoder adapts

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, len(history["train_loss"]) + 1)

# --- Loss curves ---
ax1.plot(epochs_range, history["train_loss"], label="Train loss", color="#2196F3")
ax1.plot(epochs_range, history["val_loss"], label="Val loss", color="#F44336")
ax1.axvline(x=stage_boundary + 0.5, color="gray", linestyle="--", alpha=0.7, label="Unfreeze boundary")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss (BCE + Dice)")
ax1.set_title("Training and Validation Loss")
ax1.legend()
ax1.grid(True, alpha=0.3)

# --- F1 curves ---
ax2.plot(epochs_range, history["val_f1"], label="Val F1", color="#4CAF50", linewidth=2)
ax2.plot(epochs_range, history["val_precision"], label="Val Precision", color="#FF9800", alpha=0.7)
ax2.plot(epochs_range, history["val_recall"], label="Val Recall", color="#9C27B0", alpha=0.7)
ax2.axvline(x=stage_boundary + 0.5, color="gray", linestyle="--", alpha=0.7, label="Unfreeze boundary")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Score")
ax2.set_title("Validation Metrics")
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_ylim(0, 1)

fig.suptitle("Siamese U-Net Fine-Tuning: Two-Stage Training", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(DATA_DIR / "results" / "siamese_training_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved training curves.")

## Log to MLflow

Every run logged to the same MLflow experiment so we can compare
pretrained (notebook 08) vs fine-tuned performance side by side.
Open http://localhost:5000 to see the tracking UI.

In [ ]:
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("marshall-fire-damage")

with mlflow.start_run(run_name="siamese_unet_finetuned"):
    # Log parameters
    mlflow.log_params({
        "model": "siamese_unet",
        "backbone": "resnet50",
        "patch_size": PATCH_SIZE,
        "stage1_epochs": STAGE1_EPOCHS,
        "stage2_epochs": STAGE2_EPOCHS,
        "stage1_lr": 1e-3,
        "stage2_lr": 1e-4,
        "pos_weight": 3.4,
        "batch_size": BATCH_SIZE,
        "loss": "bce_dice_combined",
        "scheduler": "cosine_annealing",
        "early_stopping_patience": patience,
        "train_parcels": len(parcels_train),
        "val_parcels": len(parcels_val),
        "spatial_split": "west_train_east_val",
        "fine_tuned": True,
    })

    # Log per-epoch metrics
    for i in range(len(history["train_loss"])):
        mlflow.log_metrics(
            {
                "train_loss": history["train_loss"][i],
                "val_loss": history["val_loss"][i],
                "val_f1": history["val_f1"][i],
                "val_precision": history["val_precision"][i],
                "val_recall": history["val_recall"][i],
            },
            step=i + 1,
        )

    # Log best metrics
    mlflow.log_metrics({
        "best_val_f1": best_val_f1,
        "best_val_precision": max(history["val_precision"]),
        "best_val_recall": max(history["val_recall"]),
    })

    # Log training curves figure
    mlflow.log_figure(fig, "training_curves.png")

    # Log best model weights
    mlflow.log_artifact(str(best_model_path), artifact_path="model")

    print("MLflow run logged.")
    print(f"  Best val F1: {best_val_f1:.4f}")
    print(f"  Run ID: {mlflow.active_run().info.run_id}")
    print("  View at: http://localhost:5000")